In [2]:
import pandas as pd
import numpy as np
import optuna
from plotly.io import show
import optuna.visualization as ov
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.inspection import DecisionBoundaryDisplay
import re

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#importazione dei dataset
df = pd.read_csv("train_titanic.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
df = df.drop(columns=['Cabin', 'Name'])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    object 
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       891 non-null    object 
 8   Fare         891 non-null    float64
 9   Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(3)
memory usage: 69.7+ KB


In [6]:
def extract_numeric(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    m = re.findall(r"[-+]?\d*\.?\d+", s)
    if not m:
        return np.nan
    return float(m[0])

numeric_cols_raw = [
    "Ticket",
]

for col in numeric_cols_raw:
    if col in df.columns:
        df[col] = df[col].apply(extract_numeric)
        
df["Age"] = df["Age"].fillna(df["Age"].median())

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    object 
 4   Age          891 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       887 non-null    float64
 8   Fare         891 non-null    float64
 9   Embarked     889 non-null    object 
dtypes: float64(3), int64(5), object(2)
memory usage: 69.7+ KB


In [7]:
#rimozione outlier
numeric_cols = df.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = df[col].quantile(0.2)
    Q3 = df[col].quantile(0.8)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    df[col] = np.where(df[col].between(lower, upper), df[col], np.nan)

df = df.dropna()

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 741 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  741 non-null    float64
 1   Survived     741 non-null    float64
 2   Pclass       741 non-null    float64
 3   Sex          741 non-null    object 
 4   Age          741 non-null    float64
 5   SibSp        741 non-null    float64
 6   Parch        741 non-null    float64
 7   Ticket       741 non-null    float64
 8   Fare         741 non-null    float64
 9   Embarked     741 non-null    object 
dtypes: float64(8), object(2)
memory usage: 63.7+ KB


In [8]:
df = pd.get_dummies(df, drop_first=True)

In [9]:
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000), # Intero tra 100 e 1000
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True), # Scala logaritmica
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0), # % feature per albero
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True), # L1 Reg (Lasso)
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True), # L2 Reg (Ridge)
        'n_jobs': 3,
        'random_state': 42
    }
    
    model = xgb.XGBClassifier(**params)
    
    # Usiamo 5-Fold per essere sicuri che il risultato non sia fortuna
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    
    if trial.should_prune():
            raise optuna.TrialPruned()
    
    # Restituiamo la media dell'accuratezza
    return scores.mean()

In [10]:
X = df.drop('Survived', axis=1)
y = df["Survived"].astype(int)

In [11]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

**OPTUNA**

In [16]:
sampler = optuna.samplers.TPESampler(seed=10)
study = optuna.create_study(direction="maximize", sampler = sampler,
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5, n_warmup_steps=30, interval_steps=10
    ),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n--- Risultati Ottimizzazione ---")
print(f"Miglior Trial (Tentativo #{study.best_trial.number})")
print(f"Accuratezza Migliore (CV): {study.best_value:.4f}")
print("Migliori Iperparametri:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

best_params = study.best_params
final_model = xgb.XGBClassifier(**best_params, n_jobs=3, random_state=42)
final_model.fit(X_train, y_train)
final_acc = final_model.score(X_test, y_test)

print(f"\nAccuratezza Finale sul Test Set: {final_acc:.4f}")

[I 2025-12-05 12:56:10,484] A new study created in memory with name: no-name-dd68316e-c214-47fc-87dd-d28e5d290bf9
Best trial: 0. Best value: 0.810825:   2%|▏         | 1/50 [00:02<02:09,  2.64s/it]

[I 2025-12-05 12:56:13,129] Trial 0 finished with value: 0.8108246688505911 and parameters: {'n_estimators': 794, 'learning_rate': 0.00112565445587888, 'max_depth': 8, 'subsample': 0.8744019412693059, 'colsample_bytree': 0.7492535061512953, 'reg_alpha': 1.0547992438188775e-06, 'reg_lambda': 6.061300044367956e-07}. Best is trial 0 with value: 0.8108246688505911.


Best trial: 0. Best value: 0.810825:   4%|▍         | 2/50 [00:03<01:21,  1.69s/it]

[I 2025-12-05 12:56:14,151] Trial 1 finished with value: 0.8056971941318901 and parameters: {'n_estimators': 785, 'learning_rate': 0.00262366298138214, 'max_depth': 3, 'subsample': 0.8426799091838986, 'colsample_bytree': 0.9766966730974682, 'reg_alpha': 1.08526150100961e-08, 'reg_lambda': 0.0004071274363191098}. Best is trial 0 with value: 0.8108246688505911.


Best trial: 0. Best value: 0.810825:   6%|▌         | 3/50 [00:05<01:27,  1.87s/it]

...

[I 2025-12-05 12:57:00,179] Trial 47 finished with value: 0.8057399230878792 and parameters: {'n_estimators': 277, 'learning_rate': 0.049554857272946895, 'max_depth': 8, 'subsample': 0.6038254545065727, 'colsample_bytree': 0.8905824861672244, 'reg_alpha': 0.030306797583127643, 'reg_lambda': 5.669934907283949e-08}. Best is trial 44 with value: 0.8226321036889331.


Best trial: 44. Best value: 0.822632:  98%|█████████▊| 49/50 [00:50<00:00,  1.41it/s]

[I 2025-12-05 12:57:00,620] Trial 48 finished with value: 0.8192565161657883 and parameters: {'n_estimators': 147, 'learning_rate': 0.01405297404705908, 'max_depth': 7, 'subsample': 0.6479514962382967, 'colsample_bytree': 0.7490359022770384, 'reg_alpha': 4.179908061802853e-08, 'reg_lambda': 3.028950986554105e-08}. Best is trial 44 with value: 0.8226321036889331.


Best trial: 44. Best value: 0.822632: 100%|██████████| 50/50 [00:50<00:00,  1.01s/it]

[I 2025-12-05 12:57:01,088] Trial 49 finished with value: 0.8074490813274462 and parameters: {'n_estimators': 157, 'learning_rate': 0.009125766855110273, 'max_depth': 7, 'subsample': 0.5531558614083414, 'colsample_bytree': 0.7218307983749165, 'reg_alpha': 7.877410602857414e-08, 'reg_lambda': 3.7729241933368674e-08}. Best is trial 44 with value: 0.8226321036889331.

--- Risultati Ottimizzazione ---
Miglior Trial (Tentativo #44)
Accuratezza Migliore (CV): 0.8226
Migliori Iperparametri:
  n_estimators: 164
  learning_rate: 0.015378390533926147
  max_depth: 8
  subsample: 0.6459385761190881
  colsample_bytree: 0.7621269752553935
  reg_alpha: 6.86748897958525e-08
  reg_lambda: 2.1786988060483554e-08

Accuratezza Finale sul Test Set: 0.8523


Senza pruner: Accuratezza Finale sul Test Set: 0.8456

Con pruner: --- Risultati Ottimizzazione ---
- Miglior Trial (Tentativo #26)
- Accuratezza Migliore (CV): 0.8159
- Migliori Iperparametri:
  - n_estimators: 380
  - learning_rate: 0.005295232368540756
  - max_depth: 7
  - subsample: 0.6891247712682177
  - colsample_bytree: 0.6752990656146991
  - reg_alpha: 0.06532885224059053
  - reg_lambda: 0.00013423432607369992

- Accuratezza Finale sul Test Set: 0.8591

Outlier 0.2, 0.8: --- Risultati Ottimizzazione ---
- Miglior Trial (Tentativo #35)
- Accuratezza Migliore (CV): 0.8192
- Migliori Iperparametri:
  - n_estimators: 821
  - learning_rate: 0.0040452630812914565
  - max_depth: 6
  - subsample: 0.8144802291812205
  - colsample_bytree: 0.8504236185894885
  - reg_alpha: 0.0020024465144113522
  - reg_lambda: 0.013159768501168314

- Accuratezza Finale sul Test Set: 0.8523

**GRAFICO**

In [22]:
ov.plot_optimization_history(study).show()
ov.plot_param_importances(study).show()
ov.plot_parallel_coordinate(study).show()
ov.plot_contour(study).show()
ov.plot_intermediate_values(study).show()

[W 2025-12-05 13:01:28,723] You need to set up the pruning feature to utilize `plot_intermediate_values()`
